# Lab Week 9 — Transformers for Neural Machine Translation

**Task:** Compare three different models for English → Spanish neural machine translation 
1. RNN 
2. Encoder-Decoder with Attention 
3. Transformer with Multi-head attention and positional encoding


We process the data using below steps: 
1. Data preparation for an English–Spanish translation dataset.
2. Text vectorization for source and target languages.
3. Transformer encoder–decoder implementation.
4. Training and validation.
5. Translation test for:
> “I like soccer and also going to the beach with friends”


### Prepare your virtual environment if you don't have one
To avoid dependency conflicts, it is recommended to use a virtual environment to install and manage the project's dependencies. You can use the **venv** module to create a virtual environment:

``` bash
python3 -m venv .venv
source .venv/bin/activate  # On Linux/MacOS
.venv\Scripts\activate     # On Windows
```

Don't forget to update your python into the latest version.

``` bash
python3 -m pip install --upgrade pip
```

**if the virtual enviroment not working:**
change the environment： 
Press: Ctrl + Shift + P
then type: "Python: Select Interpreter" 
select: .venv ... 

**Make sure your virtual enviroment has been selected**
in the .ipynb file:
1. Click the kernel name at the top right.
2. Choose Select Another Kernel...
3. Pick Python (.venv LabNLP).

In [2]:
# Make sure you use virtual environment 
import sys
print(sys.executable)

/home/user/ai-course/week9/.venv/bin/python


In [3]:
# Make sure the version of Python is 3.7 or above:
assert sys.version_info >= (3, 7)

**Warning**: the latest TensorFlow versions are based on Keras 3. For previous chapters, it wasn't too hard to update the code to support Keras 3, but unfortunately it's much harder for this chapter: for example, stateful RNNs work very differently, ragged tensors are no longer supported, TensorFlow Hub models are no longer supported, and more. So for this chapter I've had to revert to Keras 2. To do that, I set the `TF_USE_LEGACY_KERAS` environment variable to `"1"` and import the `tf_keras` package. This ensures that `tf.keras` points to `tf_keras`, which is Keras 2.*.

In [4]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tf_keras

2026-06-07 23:06:13.237913: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-07 23:06:13.337788: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-07 23:06:13.338424: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-07 23:06:13.488386: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-07 23:06:14.475111: W tensorflow/compiler/tf

And TensorFlow ≥ 2.8:

In [101]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.16.2


As we did in earlier chapters, let's define the default font sizes to make the figures prettier:

In [102]:
#pip install matplotlib
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

And let's create the `images/nlp` folder (if it doesn't already exist), and define the `save_fig()` function which is used through this notebook to save the figures in high-res for the book:

In [103]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "nlp"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

This chapter can be very slow without a GPU, so let's make sure there's one, or else issue a warning:

In [104]:
if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. Neural nets can be very slow without a GPU.")
    if "google.colab" in sys.modules:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if "kaggle_secrets" in sys.modules:
        print("Go to Settings > Accelerator and select GPU.")

No GPU was detected. Neural nets can be very slow without a GPU.


## Data Preparing

### 1. Downloads the Spanish-English dataset

In [105]:
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, cache_dir="datasets",
                               extract=True)
text = (Path(path).with_name("spa-eng") / "spa.txt").read_text()


In [106]:
#Let's see the first 20 characters of the text.
print(text[:100])
text[:100]

#\n separates different sentence pairs.
#\t separates the two sentences inside each pair.


Go.	Ve.
Go.	Vete.
Go.	Vaya.
Go.	Váyase.
Hi.	Hola.
Run!	¡Corre!
Run.	Corred.
Who?	¿Quién?
Fire!	¡Fueg


'Go.\tVe.\nGo.\tVete.\nGo.\tVaya.\nGo.\tVáyase.\nHi.\tHola.\nRun!\t¡Corre!\nRun.\tCorred.\nWho?\t¿Quién?\nFire!\t¡Fueg'

### 2.1 Text vectorization for source and target languages.
The English and Spanish text must be converted into integer token sequences.

Important implementation points:
- Keep `[start]` and `[end]` tokens in Spanish.
- Lowercase and remove most punctuation.
- Use fixed sequence lengths for batching.

**Use the 'pairs' to store the data shown below:**
```bash
[
    [English sentence, Translated sentence],
    [English sentence, Translated sentence],
    [English sentence, Translated sentence]
]
```

In [107]:
import numpy as np

text = text.replace("¡", "").replace("¿", "")

#Split a large text into lines, then split each line by tab \t, and store the result as a list called pairs.
pairs = [line.split("\t") for line in text.splitlines()]
print(pairs[:100])

np.random.seed(42)  # extra code – ensures reproducibility on CPU
np.random.shuffle(pairs)
sentences_en, sentences_es = zip(*pairs)  # separates the pairs into 2 lists

#Check the sentences in English and Spanish to verify that they are correctly paired.
for i in range(3):
    print(sentences_en[i], "=>", sentences_es[i])

[['Go.', 'Ve.'], ['Go.', 'Vete.'], ['Go.', 'Vaya.'], ['Go.', 'Váyase.'], ['Hi.', 'Hola.'], ['Run!', 'Corre!'], ['Run.', 'Corred.'], ['Who?', 'Quién?'], ['Fire!', 'Fuego!'], ['Fire!', 'Incendio!'], ['Fire!', 'Disparad!'], ['Help!', 'Ayuda!'], ['Help!', 'Socorro! Auxilio!'], ['Help!', 'Auxilio!'], ['Jump!', 'Salta!'], ['Jump.', 'Salte.'], ['Stop!', 'Parad!'], ['Stop!', 'Para!'], ['Stop!', 'Pare!'], ['Wait!', 'Espera!'], ['Wait.', 'Esperen.'], ['Go on.', 'Continúa.'], ['Go on.', 'Continúe.'], ['Hello!', 'Hola.'], ['I ran.', 'Corrí.'], ['I ran.', 'Corría.'], ['I try.', 'Lo intento.'], ['I won!', 'He ganado!'], ['Oh no!', 'Oh, no!'], ['Relax.', 'Tomátelo con soda.'], ['Smile.', 'Sonríe.'], ['Attack!', 'Al ataque!'], ['Attack!', 'Atacad!'], ['Get up.', 'Levanta.'], ['Go now.', 'Ve ahora mismo.'], ['Got it!', 'Lo tengo!'], ['Got it?', 'Lo pillas?'], ['Got it?', 'Entendiste?'], ['He ran.', 'Él corrió.'], ['Hop in.', 'Métete adentro.'], ['Hug me.', 'Abrázame.'], ['I fell.', 'Me caí.'], ['I know

In [108]:
# Set the maximum vocabulary size and maximum length of each sentence after vectorization.
vocab_size = 1000 #The TextVectorization layer will keep only the 1000 most frequent words/tokens. We use 50,000 in the class example.
max_length = 50  #Shorter sentences will be padded; longer sentences will be truncated.

# TextVectorization() can convert English text to fixed-length token ID sequences.
# Each output sequence will also have length 50.
# adapt() builds the vocabulary and assigns IDs to frequent words.
# Later, Embedding will convert these token IDs into vectors.
text_vec_layer_en = tf.keras.layers.TextVectorization(    
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=max_length)
text_vec_layer_en.adapt(sentences_en)


# Similarly, we adapt the Spanish TextVectorization layer to the Spanish sentences.
# We add "startofseq" and "endofseq" to each Spanish sentence because the decoder needs to learn when to start and when to stop generating.
text_vec_layer_es = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size, 
    output_mode="int",
    output_sequence_length=max_length)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es])

In [109]:
eng_vocab = text_vec_layer_en.get_vocabulary()
spa_vocab = text_vec_layer_es.get_vocabulary()

print("English vocab size:", len(eng_vocab))
print("Spanish vocab size:", len(spa_vocab))
print("First 20 English tokens:", eng_vocab[:20])
print("First 20 Spanish tokens:", spa_vocab[:20])

English vocab size: 1000
Spanish vocab size: 1000
First 20 English tokens: ['', '[UNK]', 'the', 'i', 'to', 'you', 'tom', 'a', 'is', 'he', 'in', 'of', 'that', 'it', 'was', 'do', 'have', 'this', 'me', 'my']
First 20 Spanish tokens: ['', '[UNK]', 'startofseq', 'endofseq', 'de', 'que', 'a', 'no', 'tom', 'la', 'el', 'en', 'es', 'un', 'me', 'se', 'por', 'lo', 'una', 'su']


### 2.2 Prepare Training and Validation Dataset

Note: 
1. When you ouput X_train, you find "b" is begining for each sentence, which means means this is a bytes string, not a normal Python Unicode string.
And if you find some output looks like "Qu\xc3\xa9", that means TensorFlow is displaying the string internally as UTF-8 encoded bytes.
For example: "Qué" will be printed as "b'Qu\xc3\xa9"

2. The output vector represents a sentence of length 50. For example: [49 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]; where 49 corresponds to "go", 1 is an [UNK] (unknown/out-of-vocabulary) word.

In [121]:
#train_size = 100_000
#valid_size = 18_964
train_size = 5000
valid_size = 1000

# English input sentences for training and validation
X_train = tf.constant(sentences_en[:train_size])
X_valid = tf.constant(sentences_en[train_size:train_size + valid_size])
print("Shape of X_Train:", X_train.shape, "\nThe first three of X_train", X_train[:3])
print("Shape of X_Valid:", X_valid.shape, "\nThe first three of X_valid", X_valid[:3])
#Shape (m,n); m = number of training examples / sentence; n = length of each sentence after vectorization

# Spanish decoder input for training and validation
# Add startofseq because the decoder needs a starting token
X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:train_size]])
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[train_size:train_size + valid_size]])
print("Shape of X_Train_Dec:", X_train_dec.shape, "\nThe first three of X_train_dec", X_train_dec[:3])
print("Shape of X_Valid_Dec:", X_valid_dec.shape, "\nThe first three of X_valid_dec", X_valid_dec[:3])

# Spanish target output for training and validation
# Add endofseq because the model should learn when to stop
# Y_train and Y_valid are decoder_output
Y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:train_size]])
Y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[train_size:train_size + valid_size]])

#As we add "startofseq" and "endofseq" to the Spanish sentences, the decoder input and target are different. 
#target is right-shifted by one token compared to the decoder input.
print("Shape of decoder_inputs_train:", decoder_inputs_train.shape, "\nThe first three of decoder_inputs_train", decoder_inputs_train[:3])
print("Shape of Y_train:", Y_train.shape, "\nThe first three of Y_train", Y_train[:3])


Shape of X_Train: (5000,) 
The first three of X_train tf.Tensor([b'How boring!' b'I love sports.' b'Would you like to swap jobs?'], shape=(3,), dtype=string)
Shape of X_Valid: (1000,) 
The first three of X_valid tf.Tensor(
[b'Tom shook hands with Mary.' b'Tell me what you want for Christmas.'
 b'We have to get up early tomorrow morning.'], shape=(3,), dtype=string)
Shape of X_Train_Dec: (5000,) 
The first three of X_train_dec tf.Tensor(
[b'startofseq Qu\xc3\xa9 aburrimiento!' b'startofseq Adoro el deporte.'
 b'startofseq Te gustar\xc3\xada que intercambiemos los trabajos?'], shape=(3,), dtype=string)
Shape of X_Valid_Dec: (1000,) 
The first three of X_valid_dec tf.Tensor(
[b'startofseq Tom y Mary se dieron un apret\xc3\xb3n de manos.'
 b'startofseq Dime lo que quieres por Navidades.'
 b'startofseq Nos tenemos que levantar temprano ma\xc3\xb1ana a la ma\xc3\xb1ana.'], shape=(3,), dtype=string)
Shape of decoder_inputs_train: (5000, 50) 
The first three of decoder_inputs_train tf.Tensor(


### 2.3. Prepare Encoder/ Decoder input and output

For each sentence pair:
- Encoder input = English token sequence.
- Decoder input = Spanish tokens except the final token.
- Decoder target = Spanish tokens except the first token.



In [ ]:
# Convert Sentences into ID sequences using the TextVectorization layers. 
# The output will be a 2D tensor of shape (m, n), where m is the number of sentences and n is the length of each sentence after vectorization (50 in this case). 
# Each element in the output tensor is an integer representing the token ID of a word in the sentence.

#encoder_inputs_train = text_vec_layer_en(X_train)
#encoder_inputs_valid = text_vec_layer_en(X_valid)
#decoder_inputs_train = text_vec_layer_es(X_train_dec)
#decoder_inputs_valid = text_vec_layer_es(X_valid_dec)



Shape of decoder_inputs_train: (5000, 50) 
The first three of decoder_inputs_train tf.Tensor(
[[  2  25   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0]
 [  2   1  10   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0]
 [  2  28 170   5   1  21   1   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0]], shape=(3, 50), dtype=int64)
Shape of Y_train: (5000, 50) 
The first three of Y_train tf.Tensor(
[[ 25   1   3   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0  

## LSTM Model

### Build a simple encoder-decoder LSTM translation model.

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility on CPU

# Define the input layer for the encoder and decoder
# shape=[] means each input sample is a single string, not a sequence of numbers yet.
# decoder_input is raw Spanish text, usually shifted right during training.
encoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)
decoder_inputs = tf.keras.layers.Input(shape=[], dtype=tf.string)

# Embedding Vector Size: Each token ID will be converted into a 128-dimensional vector.
embed_size = 128

# Convert raw English or Spanish text into token IDs: text => token ID   
# Example:
# "I love cats" -> [12, 45, 87]
encoder_input_ids = text_vec_layer_en(encoder_inputs)
decoder_input_ids = text_vec_layer_es(decoder_inputs)

# Create the embedding layer for the encoder. text => token ID => vector
# mask_zero=True: ignore padding tokens with ID 0.
encoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)
decoder_embedding_layer = tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True)

# Apply the encoder embedding layer to the English token IDs.
# Shape changes roughly from:
# (batch_size, sequence_length)
# to:
# (batch_size, sequence_length, embed_size)
encoder_embeddings = encoder_embedding_layer(encoder_input_ids)
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)

In [126]:
# The encoder LSTM reads the input sentence and stores its meaning in final states; 
# the decoder LSTM uses those states plus previous target words to predict the next target word at every time step.

# English sentence embeddings → encoder LSTM compresses meaning into states → decoder LSTM uses those states to generate Spanish sequence → Dense + softmax predicts each Spanish word.

#1. Encoder LSTM processes the English sentence embeddings and produces final states that capture the meaning of the input sentence.
encoder = tf.keras.layers.LSTM(512, return_state=True)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

#2. Decoder LSTM takes the Spanish sentence embeddings and the encoder states as initial state, and produces output sequences that will be used to predict the next Spanish word at each time step.
decoder = tf.keras.layers.LSTM(512, return_sequences=True)
decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

#3. Dense layer with softmax activation to predict the next Spanish word at each time step based on the decoder outputs.
output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(decoder_outputs)

**Warning**: the following cell will take a while to run (possibly a couple hours if you are not using a GPU).

In [ ]:
#epochs：How many times the training data was learnt
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs], 
                       outputs=[Y_proba])

model.compile(loss="sparse_categorical_crossentropy", 
              optimizer="nadam",
              metrics=["accuracy"])
model.fit(
    (X_train, X_train_dec), 
    Y_train, epochs=10,
    validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/5
157/157 [==============================] - 176s 1s/step - loss: 4.5056 - accuracy: 0.2674 - val_loss: 4.1362 - val_accuracy: 0.2997
Epoch 2/5
157/157 [==============================] - 163s 1s/step - loss: 3.9838 - accuracy: 0.3068 - val_loss: 3.8756 - val_accuracy: 0.3175
Epoch 3/5
157/157 [==============================] - 145s 921ms/step - loss: 3.6768 - accuracy: 0.3336 - val_loss: 3.6065 - val_accuracy: 0.3467
Epoch 4/5
157/157 [==============================] - 159s 1s/step - loss: 3.4132 - accuracy: 0.3588 - val_loss: 3.4656 - val_accuracy: 0.3616
Epoch 5/5
157/157 [==============================] - 164s 1s/step - loss: 3.2233 - accuracy: 0.3725 - val_loss: 3.3808 - val_accuracy: 0.3620


In [122]:
def translate(sentence_en):
    translation = ""
    for word_idx in range(max_length):
        X = np.array([sentence_en])  # encoder input 
        X_dec = np.array(["startofseq " + translation])  # decoder input
        y_proba = model.predict((X, X_dec))[0, word_idx]  # last token's probas
        predicted_word_id = np.argmax(y_proba)
        predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]
        if predicted_word == "endofseq":
            break
        translation += " " + predicted_word
    return translation.strip()

In [123]:
translate("I like soccer")

1/1 [==============================] - 0s 45ms/step


'no [UNK]'

Nice! However, the model struggles with longer sentences:

In [124]:
translate("I like soccer and also going to the beach")

1/1 [==============================] - 0s 40ms/step


'no [UNK] que [UNK] a la escuela'

## Attention Mechanisms

We need to feed all the encoder's outputs to the `Attention` layer, so we must add `return_sequences=True` to the encoder:

In [130]:
#LSTM without attention: 
#encoder = tf.keras.layers.LSTM(512, return_state=True)
#encoder_outputs, *encoder_state = encoder(encoder_embeddings)

#decoder = tf.keras.layers.LSTM(512, return_sequences=True)
#decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_state)

#output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
#Y_proba = output_layer(decoder_outputs)

tf.random.set_seed(42)  # extra code – ensures reproducibility on CPU
encoder = tf.keras.layers.LSTM(
    512,
    return_sequences=True,
    return_state=True
)
encoder_outputs, *encoder_state = encoder(encoder_embeddings)

decoder = tf.keras.layers.LSTM(512,return_sequences=True)
decoder_outputs = decoder(decoder_embeddings,initial_state=encoder_state)

# add the `Attention` layer and the output layer:
attention_layer = tf.keras.layers.Attention()
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

output_layer = tf.keras.layers.Dense(vocab_size, activation="softmax")
Y_proba = output_layer(attention_outputs)

**Warning**: the following cell will take a while to run (possibly a couple hours if you are not using a GPU).

In [129]:
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs],
                       outputs=[Y_proba])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
model.fit((X_train, X_train_dec), Y_train, epochs=10,
          validation_data=((X_valid, X_valid_dec), Y_valid))

Epoch 1/10


W0000 00:00:1778796417.394228  889349 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "Softmax" attr { key: "T" value { type: DT_FLOAT } } inputs { dtype: DT_FLOAT shape { unknown_rank: true } } device { type: "CPU" vendor: "GenuineIntel" model: "110" frequency: 2300 num_cores: 16 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 32768 l2_cache_size: 262144 l3_cache_size: 16777216 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { unknown_rank: true } }


 65/157 [===========>..................] - ETA: 1:24 - loss: 5.0635 - accuracy: 0.1635

KeyboardInterrupt: 

In [ ]:
translate("I like soccer and also going to the beach")

## Attention Is All You Need: The Transformer Architecture

The Transformer uses:
1. **Token embeddings** to represent words/subwords.
2. **Positional encoding** to represent word order.
3. **Multi-head self-attention** in the encoder.
4. **Masked multi-head self-attention** in the decoder.
5. **Cross-attention** from decoder to encoder outputs.
6. Feed-forward layers, residual connections, and layer normalization.

### Positional encodings
Transformer has no recurrence.
So it does not naturally know token order.
Therefore we add position information to word embeddings.

In [ ]:
max_length = 50  # max length in the whole training set
embed_size = 128
tf.random.set_seed(42)  # ensures reproducibility on CPU

#Keras creates a trainable weight matrix with shape (max_length, embed_size) - position embedding table 
pos_embed_layer = tf.keras.layers.Embedding(max_length, embed_size)


###Testing the position embedding layer
position_ids = tf.range(max_length)
position_embeddings = pos_embed_layer(position_ids)

print("Position IDs:",tf.range(max_length))
print("Position embedding shape:",position_embeddings.shape)
print("Position 0 embedding:", position_embeddings[0])

Position IDs: tf.Tensor(
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49], shape=(50,), dtype=int32)
Position embedding shape: (50, 128)
Position 0 embedding: tf.Tensor(
[-0.01067953 -0.0076404  -0.04898642 -0.02555643  0.02383495 -0.00865513
  0.02333121  0.00268861 -0.01728038 -0.02017183 -0.02487197 -0.04078355
  0.03462918 -0.00793942  0.00825833 -0.03580208 -0.03989273 -0.0189119
 -0.01949675  0.02186859  0.02640912 -0.03432601 -0.00714491  0.0377403
  0.0068684   0.00296292 -0.03569318  0.01263401  0.00682271 -0.04845427
  0.01853955  0.00680976  0.00904437  0.0011302   0.00781161  0.00880713
 -0.0365154   0.01954203  0.02134215  0.04330835  0.01372503  0.043995
  0.0033016  -0.03579132  0.04993173 -0.03991038 -0.04534823 -0.01911516
  0.04500834  0.03538407 -0.03650397 -0.00028207  0.00242387  0.04474989
  0.00221924 -0.02301543 -0.04492929  0.01834101  0.03349659 -0.01613039

In [138]:
# encoder_embeddings shape: (batch_size, sequence_length, embed_size)
# batch_size = how many sentences in one batch
# sequence_length = how many token position in one sentence  
# embed_size   = how many dimentions for each token  
batch_max_len_enc = tf.shape(encoder_embeddings)[1]
encoder_in = encoder_embeddings + pos_embed_layer(tf.range(batch_max_len_enc))

batch_max_len_dec = tf.shape(decoder_embeddings)[1]
decoder_in = decoder_embeddings + pos_embed_layer(tf.range(batch_max_len_dec))


Alternatively, we can use fixed, non-trainable positional encodings:

In [ ]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_length, embed_size, dtype=tf.float32, **kwargs):
        super().__init__(dtype=dtype, **kwargs)
        assert embed_size % 2 == 0, "embed_size must be even"
        p, i = np.meshgrid(np.arange(max_length),
                           2 * np.arange(embed_size // 2))
        pos_emb = np.empty((1, max_length, embed_size))
        pos_emb[0, :, ::2] = np.sin(p / 10_000 ** (i / embed_size)).T
        pos_emb[0, :, 1::2] = np.cos(p / 10_000 ** (i / embed_size)).T
        self.pos_encodings = tf.constant(pos_emb.astype(self.dtype))
        self.supports_masking = True

    def call(self, inputs):
        batch_max_length = tf.shape(inputs)[1]
        return inputs + self.pos_encodings[:, :batch_max_length]

In [ ]:
pos_embed_layer = PositionalEncoding(max_length, embed_size)
encoder_in = pos_embed_layer(encoder_embeddings)
decoder_in = pos_embed_layer(decoder_embeddings)

In [ ]:
# extra code – this cells generates and saves Figure 16–9
figure_max_length = 201
figure_embed_size = 512
pos_emb = PositionalEncoding(figure_max_length, figure_embed_size)
zeros = np.zeros((1, figure_max_length, figure_embed_size), np.float32)
P = pos_emb(zeros)[0].numpy()
i1, i2, crop_i = 100, 101, 150
p1, p2, p3 = 22, 60, 35
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(9, 5))
ax1.plot([p1, p1], [-1, 1], "k--", label="$p = {}$".format(p1))
ax1.plot([p2, p2], [-1, 1], "k--", label="$p = {}$".format(p2), alpha=0.5)
ax1.plot(p3, P[p3, i1], "bx", label="$p = {}$".format(p3))
ax1.plot(P[:,i1], "b-", label="$i = {}$".format(i1))
ax1.plot(P[:,i2], "r-", label="$i = {}$".format(i2))
ax1.plot([p1, p2], [P[p1, i1], P[p2, i1]], "bo")
ax1.plot([p1, p2], [P[p1, i2], P[p2, i2]], "ro")
ax1.legend(loc="center right", fontsize=14, framealpha=0.95)
ax1.set_ylabel("$P_{(p,i)}$", rotation=0, fontsize=16)
ax1.grid(True, alpha=0.3)
ax1.hlines(0, 0, figure_max_length - 1, color="k", linewidth=1, alpha=0.3)
ax1.axis([0, figure_max_length - 1, -1, 1])
ax2.imshow(P.T[:crop_i], cmap="gray", interpolation="bilinear", aspect="auto")
ax2.hlines(i1, 0, figure_max_length - 1, color="b", linewidth=3)
cheat = 2  # need to raise the red line a bit, or else it hides the blue one
ax2.hlines(i2+cheat, 0, figure_max_length - 1, color="r", linewidth=3)
ax2.plot([p1, p1], [0, crop_i], "k--")
ax2.plot([p2, p2], [0, crop_i], "k--", alpha=0.5)
ax2.plot([p1, p2], [i2+cheat, i2+cheat], "ro")
ax2.plot([p1, p2], [i1, i1], "bo")
ax2.axis([0, figure_max_length - 1, 0, crop_i])
ax2.set_xlabel("$p$", fontsize=16)
ax2.set_ylabel("$i$", rotation=0, fontsize=16)
save_fig("positional_embedding_plot")
plt.show()

### Multi-Head Attention

In [ ]:
N = 2  # instead of 6
num_heads = 8
dropout_rate = 0.1
n_units = 128  # for the first Dense layer in each Feed Forward block
encoder_pad_mask = tf.math.not_equal(encoder_input_ids, 0)[:, tf.newaxis]
Z = encoder_in
for _ in range(N):
    skip = Z
    attn_layer = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_size, dropout=dropout_rate)
    Z = attn_layer(Z, value=Z, attention_mask=encoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))
    skip = Z
    Z = tf.keras.layers.Dense(n_units, activation="relu")(Z)
    Z = tf.keras.layers.Dense(embed_size)(Z)
    Z = tf.keras.layers.Dropout(dropout_rate)(Z)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

In [ ]:
decoder_pad_mask = tf.math.not_equal(decoder_input_ids, 0)[:, tf.newaxis]
causal_mask = tf.linalg.band_part(  # creates a lower triangular matrix
    tf.ones((batch_max_len_dec, batch_max_len_dec), tf.bool), -1, 0)

In [ ]:
encoder_outputs = Z  # let's save the encoder's final outputs
Z = decoder_in  # the decoder starts with its own inputs
for _ in range(N):
    skip = Z
    attn_layer = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_size, dropout=dropout_rate)
    Z = attn_layer(Z, value=Z, attention_mask=causal_mask & decoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))
    skip = Z
    attn_layer = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_size, dropout=dropout_rate)
    Z = attn_layer(Z, value=encoder_outputs, attention_mask=encoder_pad_mask)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))
    skip = Z
    Z = tf.keras.layers.Dense(n_units, activation="relu")(Z)
    Z = tf.keras.layers.Dense(embed_size)(Z)
    Z = tf.keras.layers.LayerNormalization()(tf.keras.layers.Add()([Z, skip]))

**Warning**: the following cell will take a while to run (possibly 2 or 3 hours if you are not using a GPU).

In [ ]:
Y_proba = tf.keras.layers.Dense(vocab_size, activation="softmax")(Z)
model = tf.keras.Model(inputs=[encoder_inputs, decoder_inputs],
                       outputs=[Y_proba])
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
model.fit((X_train, X_train_dec), Y_train, epochs=10,
          validation_data=((X_valid, X_valid_dec), Y_valid))

In [ ]:
translate("I like soccer and also going to the beach")